# Convert your MNIST model from Tensorflow to ONNX and run it on UbiOps twice as fast

In this recipe we will show you how to convert a Tensorflow based image classification algorithm to ONNX and 
run it on UbiOps using the ONNX runtime.

First lets connect to UbiOps and load all of our dependencies.

In [ ]:
!pip install tensorflow==2.3.1 tf2onnx==1.9.1 tqdm==4.62.0

In [ ]:
API_TOKEN = 'Token tokentokentoken' # Fill in your token here
PROJECT_NAME = 'my-project name'    # Fill in your project name here
DEPLOYMENT_NAME = 'tf-vs-onnx'
import ubiops 
import shutil
from tensorflow.keras.models import load_model
import random, glob
import time
from datetime import datetime, timedelta
from tqdm import tqdm
import shutil


configuration = ubiops.Configuration(host="https://api.ubiops.com/v2.1")
configuration.api_key['Authorization'] = API_TOKEN

client = ubiops.ApiClient(configuration)
api = ubiops.CoreApi(client)
api.service_status()

## Converting the model

First step is to convert the model from Tensorflow to ONNX. I had the model in a `.h5` format so I had to load and save it as a `SavedModel` first. Then convert it using the `tf2onnx` package.

If you already have the model as a `SavedModel` then you can skip the first two lines.

If everything worked correctly you should have the ONNX model at ```mnist_deployment_onnx_package/mnist.onnx```.

In [ ]:
model = load_model("mnist_deployment_package/cnn.h5")
model.save("mnist_model")

# tf2onnx.convert is a cli programm
!python3 -m tf2onnx.convert --saved-model mnist_model --opset 13 --output mnist_deployment_onnx_package/mnist.onnx


## Preparing the comparison

The next step is to create two deployments one with the original Tensorflow based runtime and the second with the ONNX model runnning on the ONNX runtime. The rest of the deployments is the same.

The following code will save the deployments.py to mnist_deployment_package and mnist_deployment_onnx_package respectively.

In [ ]:
%%writefile ./mnist_deployment_package/deployment.py
"""
The file containing the deployment code is required to be called 'deployment.py' and should contain the 'Deployment'
class and 'request' method.
"""

import os
from tensorflow.keras.models import load_model
from imageio import imread
import numpy as np


class Deployment:

    def __init__(self, base_directory, context):

        print("Initialising deployment")

        weights = os.path.join(base_directory, "cnn.h5")
        self.model = load_model(weights)

    def request(self, data):

        print("Processing request")

        x = imread(data['image'])
        # convert to a 4D tensor to feed into our model
        x = x.reshape(1, 28, 28, 1)
        x = x.astype(np.float32) / 255

        out = self.model.predict(x)

        # here we set our output parameters in the form of a json
        return {'prediction': int(np.argmax(out)), 'probability': float(np.max(out))}


In [ ]:
%%writefile ./mnist_deployment_onnx_package/deployment.py
"""
The file containing the deployment code is required to be called 'deployment.py' and should contain the 'Deployment'
class and 'request' method.
"""

import os
import onnxruntime as rt
from imageio import imread
import numpy as np


class Deployment:

    def __init__(self, base_directory, context):
        self.sess = rt.InferenceSession("mnist.onnx")
        self.input_name = self.sess.get_inputs()[0].name

    def request(self, data):


        x = imread(data['image'])
        # convert to a 4D tensor to feed into our model
        x = x.reshape(1, 28, 28, 1)
        x = x.astype(np.float32) / 255

        print("Prediction being made")

        prediction = self.sess.run(None, {self.input_name: x})[0]

        return {'prediction': int(np.argmax(prediction)), 'probability': float(np.max(prediction))}

       


Now that the deployment packages are finished you can upload them to UbiOps. We will make one deployment with two versions, one running Tensorflow while the other is running ONNX.

In [ ]:
mnist_template = ubiops.DeploymentCreate(
    name=DEPLOYMENT_NAME,
    description='A deployment to classify handwritten digits.',
    input_type='structured',
    output_type='structured',
    input_fields=[
        {'name': 'image', 'data_type': 'blob'}
    ],
    output_fields=[
        {'name': 'prediction', 'data_type': 'int'},
        {'name': 'probability', 'data_type': 'double'}
    ]
)

mnist_deployment = api.deployments_create(project_name=PROJECT_NAME, data=mnist_template)
print(mnist_deployment)

In [ ]:
version_template = ubiops.DeploymentVersionCreate(
    version="onnx",
    language='python3.7',
    memory_allocation=1024,
    maximum_instances=1,
    minimum_instances=0,
    maximum_idle_time=1800, # = 30 minutes
    deployment_mode='express',  # 'express' or 'batch'
    request_retention_mode='full',  # input/output of requests will be stored
    request_retention_time=3600  # requests will be stored for 1 hour
)

version = api.deployment_versions_create(
    project_name=PROJECT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    data=version_template
)

In [ ]:
# Zip the deployment package
shutil.make_archive('mnist_deployment_onnx_package', 'zip', '.', 'mnist_deployment_onnx_package')


upload_response = api.revisions_file_upload(
    project_name=PROJECT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    version="onnx",
    file='mnist_deployment_onnx_package.zip'
)
print(upload_response)

In [ ]:
version_template = ubiops.DeploymentVersionCreate(
    version="tf",
    language='python3.7',
    memory_allocation=1024,
    maximum_instances=1,
    minimum_instances=0,
    maximum_idle_time=1800, # = 30 minutes
    deployment_mode='express',  # 'express' or 'batch'
    request_retention_mode='full',  # input/output of requests will be stored
    request_retention_time=3600  # requests will be stored for 1 hour
)

version = api.deployment_versions_create(
    project_name=PROJECT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    data=version_template
)

In [ ]:
# Zip the deployment package
shutil.make_archive('mnist_deployment_package', 'zip', '.', 'mnist_deployment_package')


upload_response = api.revisions_file_upload(
    project_name=PROJECT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    version="tf",
    file='mnist_deployment_package.zip'
)
print(upload_response)

## Benchmarking

If everything went well there should now be a deployment in UbiOps with two versions. We can now compare the average request time by sending both versions a list of 100 images (one image per request.)

In [ ]:
shutil.unpack_archive("mnist_png.zip", "./") # Unzip test images.

pattern = "mnist_png/testing/*/*.png" # (or "*.*")
filenames = random.choices(glob.glob(pattern),k=100)
print(filenames)

In [ ]:
ready = False
while not ready:   
    time.sleep(5)
    response = api.deployment_versions_list(project_name=PROJECT_NAME,
        deployment_name=DEPLOYMENT_NAME)
    statuses = [d.status == 'available' for d in response]
    ready = all(statuses)
    
    print("Deployments are NOT ready")

print("Deployments are ready")

print("Uploading test images and making requests")

for image_file in tqdm(filenames):    
    # First upload the image
    blob = api.blobs_create(project_name=PROJECT_NAME, file=image_file)
    
    # Make a request using the blob id as input.
    data = {'image': blob.id}
    
    time.sleep(.05) # Let's not crash the api    
    api.deployment_version_requests_create(
        project_name=PROJECT_NAME,
        deployment_name=DEPLOYMENT_NAME,
        version="onnx",
        data=data
    )

    api.deployment_version_requests_create(
        project_name=PROJECT_NAME,
        deployment_name=DEPLOYMENT_NAME,
        version="tf",
        data=data
    )

print("Done")

## Comparing the results

Now that the request are finished we can look at the results. You can do that either by looking in the UbiOps webapp by going to the metrics tab of both versions or by running the following piece of code.

In [ ]:

version_id = api.deployment_versions_get(PROJECT_NAME,DEPLOYMENT_NAME, "tf").id

print("Average request time (s)")

api_response = api.metrics_get(
    project_name=PROJECT_NAME,
    object_type="deployment_version",
    object_id=version_id,
    metric="compute",
    interval="day",
    start_time=str((datetime.today()- timedelta(days=1)).strftime('%Y-%m-%dT%H:%M:%SZ')),
    end_time=str(datetime.today().strftime('%Y-%m-%dT%H:%M:%SZ')),
)
print(f"Tensorflow: {api_response[-1].value}")

version_id = api.deployment_versions_get(PROJECT_NAME,DEPLOYMENT_NAME, "onnx").id


api_response = api.metrics_get(
    project_name=PROJECT_NAME,
    object_type="deployment_version",
    object_id=version_id,
    metric="compute",
    interval="day",
    start_time=str((datetime.today()- timedelta(days=1)).strftime('%Y-%m-%dT%H:%M:%SZ')),
    end_time=str(datetime.today().strftime('%Y-%m-%dT%H:%M:%SZ')),
)
print(f"ONNX:       {api_response[-1].value}")



# Cleaning up



In [ ]:
api.client_close()